# 04i | Prefix sums and an intro to dynamic programming

The two techniques in this lesson share one idea: **store what you've already computed and reuse it later.**

Dynamic programming (DP) is the problem type many people fear most. DS intern interviews only test the most basic one-dimensional DP, and the four problems in this lesson cover the common shapes.

About 55 minutes.

## 1. Prefix sums

`prefix[i]` = the sum of the first i numbers. With it, the sum of any slice `nums[i:j]` is `prefix[j] - prefix[i]`, computed in O(1) instead of re-adding every time.

```
nums   =    [3, 1, 4, 1, 5]
prefix = [0, 3, 4, 8, 9, 14]        (an extra 0 at the front keeps the formula clean)
sum of nums[1:4] = prefix[4] - prefix[1] = 9 - 3 = 6
```

This is just `cumsum`.

In [ ]:
nums = [3, 1, 4, 1, 5]
prefix = [0]
for x in nums:
    prefix.append(prefix[-1] + x)
print(prefix)
print(prefix[4] - prefix[1], sum(nums[1:4]))    # both methods give the same result

### Prefix sums + a hash map

**Problem: how many contiguous subarrays sum to exactly k?** (The array may contain negative numbers, so a sliding window won't work.)

The sum of a slice = the difference of two prefix sums. So the number of "subarrays ending at the current position with sum k" equals **how many earlier prefix sums had the value `current prefix sum − k`**.

Keep a dict of "how many times each prefix sum has appeared" and look it up as you go. This is the same pattern as Two Sum (check whether "the other half I need" has appeared before).

### ✏️ Exercise 1 (fill in the blanks)

Return the number of contiguous subarrays whose sum is exactly `k`.

In [ ]:
def subarray_sum_k(nums, k):
    count = 0
    prefix = 0
    seen = {0: 1}                 # prefix sum -> count; the sum of "nothing chosen yet" is 0
    for x in nums:
        prefix += x
        count += seen.get(prefix - ____, 0)
        seen[prefix] = seen.get(prefix, 0) + ____
    return count

print(subarray_sum_k([1, 1, 1], 2))

<details>
<summary><b>Answer</b></summary>

```python
def subarray_sum_k(nums, k):
    count = 0
    prefix = 0
    seen = {0: 1}                 # prefix sum -> count; the sum of "nothing chosen yet" is 0
    for x in nums:
        prefix += x
        count += seen.get(prefix - k, 0)
        seen[prefix] = seen.get(prefix, 0) + 1
    return count

print(subarray_sum_k([1, 1, 1], 2))
```

</details>

## 2. Dynamic programming

DP fits problems where **the answer to a big problem can be derived from answers to smaller problems**, and the smaller problems get reused over and over.

For every DP problem, ask yourself the same three questions:

1. **State:** what does `dp[i]` mean? (The most important step; usually "the best answer considering everything up to item i".)
2. **Transition:** how is `dp[i]` computed from earlier `dp` values?
3. **Initial values:** what are the answers for the smallest cases?

Then fill in the table from small to large.

### Example: climbing stairs

You can climb 1 or 2 steps at a time. How many ways are there to reach step n?

1. State: `dp[i]` = the number of ways to reach step i;
2. Transition: the last move came either from i−1 or from i−2, so `dp[i] = dp[i-1] + dp[i-2]`;
3. Initial values: `dp[1] = 1`, `dp[2] = 2`.

(It's Fibonacci. The memoization in the previous lesson is "top-down"; filling a table here is "bottom-up". The two are equivalent.)

### ✏️ Exercise 2 (write it yourself)

Return the number of ways to climb to step `n` (`n ≥ 1`, 1 or 2 steps at a time).

`climb_stairs(3)` → `3` (1+1+1, 1+2, 2+1).

In [ ]:
def climb_stairs(n):
    # write your code below
    pass

<details>
<summary><b>Hint</b></summary>

You can build a list: `dp = [0] * (n + 1)`, fill in `dp[1]` and `dp[2]` (watch out for n = 1), then `for i in range(3, n + 1): dp[i] = dp[i-1] + dp[i-2]`, and return `dp[n]`.

</details>

<details>
<summary><b>Answer</b></summary>

```python
def climb_stairs(n):
    if n <= 2:
        return n
    prev, cur = 1, 2              # dp[1], dp[2]
    for _ in range(3, n + 1):
        prev, cur = cur, prev + cur
    return cur
```

Since `dp[i]` depends only on the previous two values, two variables are enough; you don't need the whole list. In an interview, write the list version first, then mention that space can be optimized to O(1).

</details>

### Example: maximum subarray sum (Kadane's algorithm)

Find the **contiguous** subarray with the largest sum and return that sum. `[-2, 1, -3, 4, -1, 2, 1, -5, 4]` → `6` (`[4, -1, 2, 1]`).

1. State: `cur` = the largest sum of a subarray **ending at the current element**;
2. Transition: either extend the previous run (`cur + x`) or start fresh at this element (`x`). Take the larger: `cur = max(x, cur + x)`;
3. The answer is the largest `cur` seen.

Intuition: if the previous run has a negative sum, carrying it along only drags you down, so drop it and start over.

### ✏️ Exercise 3 (write it yourself)

Return the largest sum of a contiguous subarray of `nums` (the array is non-empty, and the subarray has at least one element).

In [ ]:
def max_subarray(nums):
    # write your code below
    pass

<details>
<summary><b>Hint</b></summary>

`cur = best = nums[0]`. From the second element on: `cur = max(x, cur + x)`, `best = max(best, cur)`. Don't initialize to 0, or you'll get the wrong answer when every number is negative.

</details>

<details>
<summary><b>Answer</b></summary>

```python
def max_subarray(nums):
    cur = nums[0]
    best = nums[0]
    for x in nums[1:]:
        cur = max(x, cur + x)
        best = max(best, cur)
    return best
```

</details>

### Example: house robber

A row of houses each holds some amount of money, and you **can't rob two adjacent houses**. What's the most you can steal? `[2, 7, 9, 3, 1]` → `12` (2 + 9 + 1).

1. State: `dp[i]` = the maximum amount considering only the first i houses;
2. Transition: house i has two options:
   - skip it: `dp[i-1]`;
   - rob it: `dp[i-2] + the amount in house i` (house i−1 is then off limits);
   - take the larger;
3. Initial values: `dp[0] = 0` (no houses), `dp[1]` = the amount in the first house.

"**Take it or skip it**" is the most common DP transition.

### ✏️ Exercise 4 (write it yourself)

Return the maximum amount you can rob without robbing adjacent houses. An empty list returns 0.

In [ ]:
def rob(nums):
    # write your code below
    pass

<details>
<summary><b>Hint</b></summary>

List version: `dp = [0] * (len(nums) + 1)`; `dp[1] = nums[0]` (handle the empty list first); `for i in range(2, len(nums) + 1): dp[i] = max(dp[i-1], dp[i-2] + nums[i-1])`; return `dp[-1]`. Note that the amount in house i is `nums[i-1]`.

</details>

<details>
<summary><b>Answer</b></summary>

```python
def rob(nums):
    prev, cur = 0, 0              # dp[i-2], dp[i-1]
    for x in nums:
        prev, cur = cur, max(cur, prev + x)
    return cur
```

</details>

### Example: coin change

Given several coin denominations (unlimited supply of each) and a total amount, what is the **minimum** number of coins that adds up to it? Return -1 if it can't be done. `coins = [1, 2, 5]`, `amount = 11` → `3` (5 + 5 + 1).

1. State: `dp[a]` = the minimum number of coins needed to make amount a;
2. Transition: which coin c was used last? Then before it you had made `a − c`. Take the minimum over all usable c: `dp[a] = min(dp[a − c] + 1)`;
3. Initial values: `dp[0] = 0`; set everything else to "infinity" (meaning not reachable yet).

In Python, infinity is written `float("inf")`.

### ✏️ Exercise 5 (write it yourself)

Return the minimum number of coins needed to make `amount`; return `-1` if it can't be made.

In [ ]:
def coin_change(coins, amount):
    INF = float("inf")
    dp = [0] + [INF] * amount          # dp[a]: minimum number of coins to make a
    for a in range(1, amount + 1):
        for c in coins:
            # write your code below
            pass
    return -1 if dp[amount] == INF else dp[amount]

<details>
<summary><b>Hint</b></summary>

If `c <= a` (the coin doesn't exceed the current amount), then `dp[a] = min(dp[a], dp[a - c] + 1)`.

</details>

<details>
<summary><b>Answer</b></summary>

```python
def coin_change(coins, amount):
    INF = float("inf")
    dp = [0] + [INF] * amount          # dp[a]: minimum number of coins to make a
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a:
                dp[a] = min(dp[a], dp[a - c] + 1)
    return -1 if dp[amount] == INF else dp[amount]
```

Note that with `[1, 3, 4]`, the answer for 6 is 2 (3 + 3), while "always take the largest coin" gives 4 + 1 + 1 = 3 coins. Greedy is wrong here; you need DP.

</details>

## Summary

- Range sums → prefix sums; "subarrays summing to k" → prefix sums + a dict;
- The three DP questions: what is the state, what is the transition, what are the initial values;
- Common transitions: "come from one or two steps back" (climbing stairs), "extend or restart" (maximum subarray), "take it or skip it" (house robber), "which item was used last" (coin change).

The next lesson is a timed mock. On to **04j**.